In [17]:
import json
import re
import shutil
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv

from pydantic import BaseModel

from typing import List

load_dotenv() 

True

In [18]:
str_path_pc = "C:\\Users\\lpasquarelli\\OneDrive - BUSINESS INTEGRATION PARTNERS SPA\\experiments"
OBSIDIAN_DIR = Path(str_path_pc)

In [19]:
MODEL_NAME = "gpt-5.4-mini"
#OBSIDIAN_DIR = Path(r"/Users/luco/llmwiki/")
RAW_DIR = OBSIDIAN_DIR / "chunks"
WIKI_DIR = OBSIDIAN_DIR / "AI Wiki"
PULSES_DIR = WIKI_DIR / "pulses"
PROJECTS_DIR = WIKI_DIR / "projects"
TECHNOLOGIES_DIR = WIKI_DIR / "technologies"
INDEX_FILE = WIKI_DIR / "index.md"
#prova

In [20]:
class TechnologyPulse(BaseModel):
    tech_id: str
    date: str
    new_skill: str

class ProjectPulse(BaseModel):
    project_id: str
    date: str
    current_state: str

class Pulse(BaseModel):
    pulse_id: str
    summary: str
    projects: list[ProjectPulse]
    technologies: list[TechnologyPulse]

class Pulses(BaseModel):
    pulses: list[Pulse]
    projects_list: List[str] = []
    technologies_list: List[str] = []  

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage

model = init_chat_model(
    "openai:gpt-5.4-nano"
)

structured_llm = model.with_structured_output(Pulses)

SYSTEM_PROMPT = """You are a deterministic knowledge base extractor working on daily work reports.

Your goal is to extract structured nodes that will populate a personal professional 
knowledge base tracking project evolution, technologies used, and skills demonstrated.

## Output nodes

### PULSE
One pulse per day. Captures what problem was being solved, the approach taken, 
and key technical decisions. The pulse_id must be the date in YYYY-MM-DD format.

### PROJECTS
Each project has an id (the canonical project name) and a brief description of its 
state on that day. Projects are long-lived initiatives, not one-off tasks.

### TECHNOLOGIES
Each technology has an id (the canonical technology name) and a brief description 
of the specific skill learned or applied on that day.

## Technology classification rules — read carefully

A TECHNOLOGY must be a real, reusable software tool, library, framework, platform, 
or infrastructure service that exists independently of this project.

Valid technologies: pandas, scikit-learn, FastAPI, Databricks, VS Code, 
   Azure Functions, spaCy, Delta Lake, PySpark, pytest, Docker, Git

NOT technologies — do not add these:
- Project-specific artifacts: scripts, notebooks, modules, config files, 
  custom functions, field names, or internal identifiers (e.g. "LOOKUP_DY365", 
  "string_utils.py", "validation_engine")
- Concepts or processes: "normalization", "ground truth", "evaluation", 
  "data quality", "debugging"
- Generic tool categories without specificity: "notebook", "script", "pipeline", 
  "model" — always name the actual tool (e.g. "Databricks notebook", "scikit-learn 
  pipeline", "OpenAI GPT-4")

When in doubt, ask yourself: "Can I install or import this?" If no → it is not a technology.

## Vocabulary reuse rules

The knowledge base maintains canonical vocabularies:
- projects: {projects}
- technologies: {technologies}

Rules:
1. Always prefer an existing canonical name over creating a new one.
2. Only create a new entry if the concept is genuinely absent — not a variation, 
   alias, or abbreviation of something existing.
3. When you create a new entry, include it in projects_list / technologies_list 
   so the vocabulary stays up to date.
4. Use consistent casing: PascalCase for proper tool names (Databricks, VS Code), 
   lowercase for libraries (pandas, spacy, fastapi).

## Output language

Write all summaries, current_state, and new_skill fields in the same language 
as the input report. Do not switch languages mid-output.

## Field guidance

- pulse.summary: 2–4 sentences. Cover (1) the main problem tackled, (2) the 
  approach or solution attempted, (3) any open issues or decisions deferred. 
  Avoid vague filler like "I worked on X" — be specific about what changed.
- project.current_state: one sentence describing where the project stands on 
  this day, focusing on blockers, progress, or next steps.
- technology.new_skill: one sentence on what was learned or applied, specific 
  to that day's work.
"""

def build_prompt(report: str, projects: list[str] = None, technologies: list[str] = None) -> list:
    system = SYSTEM_PROMPT.format(
        projects=json.dumps(projects or []),
        technologies=json.dumps(technologies or [])
    )
    return [
        SystemMessage(content=system),
        HumanMessage(content=report),  # raw string, never parsed for {variables}
    ]

In [22]:
def list_raw_files() -> list[Path]:
    list_files = sorted(RAW_DIR.rglob("*.md"))
    if any(list_files):
        return list_files
    else:
        raise FileNotFoundError(f"No markdown files found in {RAW_DIR}")

def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8") if path.exists() else ""

In [23]:
source_files = list_raw_files()
projects: List[str] = []
technologies: List[str] = []  

responses= []

#config = {"configurable": {"thread_id": "1234567890"}} 
for i, file in enumerate(source_files):
    content = read_text(file)
    prompt = build_prompt(content, projects, technologies)
    try:
        response = structured_llm.invoke(prompt)
        responses.append(response)
        projects.extend(response.projects_list)
        projects = list(set(projects))  # Remove duplicates
        technologies.extend(response.technologies_list)
        technologies = list(set(technologies))  # Remove duplicates
    except Exception as e:
        print(f"Error processing file {file}: {e}")
    # if i == 3:
    #     break
print(f"Processed {i+1} files")
    

Processed 14 files


In [24]:
response

Pulses(pulses=[Pulse(pulse_id='2026-04-14', summary='Ho approfondito il funzionamento del merge e dei merge conflict, in particolare su come vengono integrate le diff e su come gestire i metadati dei notebook quando sorgono conflitti. In parallelo ho chiarito la logica di team su Git, con l’idea di fare merge frequenti su dev e pull del proprio branch per evitare disallineamenti. Sul fronte funzionale ho completato la prima parte della soluzione per AGEVOLAZIONI FISCALI sui dati lista, ma resta da rivedere la logica per evitare che un singolo POD fallito faccia fallire tutto il modulo.', projects=[ProjectPulse(project_id='AGEVOLAZIONI FISCALI', date='2026-04-14', current_state='È stata completata la refactor del validation engine per supportare liste e multimodulo, ma la gestione dell’errore per singolo elemento va ancora corretta perché oggi un fallimento blocca tutto il modulo.'), ProjectPulse(project_id='OCR BATCH', date='2026-04-14', current_state='Resta da eseguire il run dei cont

In [25]:
with open("response.json", "w") as f:
    json.dump(response.model_dump(), f, indent=2)

In [26]:
# BUILDING MARKDOWN PAGES UTILITY FUNCTIONS
def render_pulse_md(pulse: Pulse):
    projects =  [p for p in pulse.projects]
    technologies = [t for t in pulse.technologies]

    render_technologies = "\n".join([f"- {t.tech_id}: {t.new_skill}" for t in technologies])
    render_projects = "\n".join([f"- {p.project_id}: {p.current_state}" for p in projects])
    return f"### {pulse.pulse_id} \n {pulse.summary} \n **Projects:** \n {render_projects} \n **Technologies:** \n {render_technologies}"


from collections import defaultdict

def init_pulses_page(page_path: Path) -> None:
    """Crea la pagina con il titolo se non esiste già."""
    if page_path.exists():
        return
    page_path.parent.mkdir(parents=True, exist_ok=True)
    title = page_path.stem  # es. "2026-01"
    page_path.write_text(f"# Pulses {title}\n\n", encoding="utf-8")

def append_pulse_to_page(page_path: Path, pulse: Pulse) -> None:
    """Appende il markdown di un pulse alla pagina."""
    with page_path.open("a", encoding="utf-8") as f:
        f.write(render_pulse_md(pulse) + "\n\n")

def write_pulses(pulses: list[Pulse], page_path: Path) -> None:
    # raggruppa per mese (YYYY-MM)
    by_month: dict[str, list[Pulse]] = defaultdict(list)
    for p in pulses:
        month_year = p.pulse_id[:7]   # "2026-01-26" -> "2026-01"
        by_month[month_year].append(p)

    for month_year, month_pulses in by_month.items():
        page = page_path / f"{month_year}.md"
        init_pulses_page(page)
        # ordina cronologicamente
        for pulse in sorted(month_pulses, key=lambda x: x.pulse_id):
            append_pulse_to_page(page, pulse)

# BUILD UTILITIES FOR PROJECTS AND TECHNOLOGIES PAGES 
def init_page(name:str, parent_path:Path):
    page_path = parent_path / f"{name}.md"
    print(page_path)
    if page_path.exists():
        print("already exists")
        return
    page_path.parent.mkdir(parents=True, exist_ok=True)
    return page_path


def list_render_md(bullet_ids: List[str],page_path: Path):
    text = "\n - ".join(bullet_ids)
    with page_path.open("a", encoding="utf-8") as f:
        f.write("\n - ")
        f.write(text)
    

def write_list(pulses: List[Pulse], file_name: str = "technologies", type:str = "technologies", parent_path: Path = WIKI_DIR):
    bullet_set = set()
    for p in pulses:
        if type == "technologies":
            pulse_list = [t.tech_id for t in p.technologies]
        elif type == "projects":
            pulse_list = [z.project_id for z in p.projects]
        bullet_set.update(pulse_list)
    bullet_list = list(bullet_set)    
    page_path = init_page(name=file_name,parent_path=parent_path)
    list_render_md(bullet_ids=bullet_list,page_path=page_path)





In [27]:
# actually write the markdown

PULSES_DIR.mkdir(parents=True, exist_ok=True)
pulses = []
for response in responses:
    pulses.extend(response.pulses)
print(pulses)
write_pulses(pulses,page_path=PULSES_DIR)
write_list(pulses, file_name="technologies", type="technologies", parent_path=WIKI_DIR)
write_list(pulses, file_name="projects", type="projects", parent_path=WIKI_DIR)

[Pulse(pulse_id='2026-01-29', summary="Ho lavorato al refactoring dell'ingestion dei documenti per rendere dinamico il riconoscimento della tipologia di modulo e collegarlo correttamente ai basemodel e ai prompt definiti nel dominio. In parallelo ho iniziato a sistemare i controlli per ccia e ateco, ma è emerso un problema nella similarità testuale per le descrizioni ATECO, con casi in cui il match falliva in modo inatteso. Ho quindi aperto una linea di bug fixing sulla logica di confronto stringhe, lasciando aperti altri errori di validazione da verificare.", projects=[ProjectPulse(project_id='document_ingestion', date='2026-01-29', current_state='Il riconoscimento dinamico del tipo documento e il mapping verso model/prompt risultano impostati e da validare su casi aggiuntivi.'), ProjectPulse(project_id='validation_engine', date='2026-01-29', current_state='Sono stati aggiunti nuovi controlli per ccia e ateco, ma la logica di similarità testuale richiede correzioni per evitare falsi n

In [28]:
pulses

[Pulse(pulse_id='2026-01-29', summary="Ho lavorato al refactoring dell'ingestion dei documenti per rendere dinamico il riconoscimento della tipologia di modulo e collegarlo correttamente ai basemodel e ai prompt definiti nel dominio. In parallelo ho iniziato a sistemare i controlli per ccia e ateco, ma è emerso un problema nella similarità testuale per le descrizioni ATECO, con casi in cui il match falliva in modo inatteso. Ho quindi aperto una linea di bug fixing sulla logica di confronto stringhe, lasciando aperti altri errori di validazione da verificare.", projects=[ProjectPulse(project_id='document_ingestion', date='2026-01-29', current_state='Il riconoscimento dinamico del tipo documento e il mapping verso model/prompt risultano impostati e da validare su casi aggiuntivi.'), ProjectPulse(project_id='validation_engine', date='2026-01-29', current_state='Sono stati aggiunti nuovi controlli per ccia e ateco, ma la logica di similarità testuale richiede correzioni per evitare falsi n

In [29]:
responses

[Pulses(pulses=[Pulse(pulse_id='2026-01-29', summary="Ho lavorato al refactoring dell'ingestion dei documenti per rendere dinamico il riconoscimento della tipologia di modulo e collegarlo correttamente ai basemodel e ai prompt definiti nel dominio. In parallelo ho iniziato a sistemare i controlli per ccia e ateco, ma è emerso un problema nella similarità testuale per le descrizioni ATECO, con casi in cui il match falliva in modo inatteso. Ho quindi aperto una linea di bug fixing sulla logica di confronto stringhe, lasciando aperti altri errori di validazione da verificare.", projects=[ProjectPulse(project_id='document_ingestion', date='2026-01-29', current_state='Il riconoscimento dinamico del tipo documento e il mapping verso model/prompt risultano impostati e da validare su casi aggiuntivi.'), ProjectPulse(project_id='validation_engine', date='2026-01-29', current_state='Sono stati aggiunti nuovi controlli per ccia e ateco, ma la logica di similarità testuale richiede correzioni per 